# Object Detection
Using the MMdetection3D package, we have a wide range of object detection networks we can use.
For our case on incomplete partial rgb scans, we use votenet

In [ ]:
import os
import sys
sys.path.insert(0, '../')
import drm

## Loading the pointcloud
The model expects bin files (xyz rgb8)
Z up. The pointclouds created by Unity have Y up, so we rotate them around the x-axis

In [ ]:
# Convert the txt to a correctly oriented bin file for votenet
txtPath = "/home/jvermandere/projects/DRM/_input/Office1 - Cloud.txt"
binPath = drm.txt_pcd_to_bin(txtPath, rotateX=True)

## Object detection
Object detection is performed using the scripts provided by MMdet3D. using the following example command:

```
python  demo/pcd_demo.py 
        demo/data/sunrgbd/000017.bin 
        configs/votenet/votenet_8xb16_sunrgbd-3d.py 
        "/home/jvermandere/projects/DRM/checkpoints/votenet_16x8_sunrgbd-3d-10class_20210820_162823-bf11f014.pth"
```

In [ ]:
detectionJsonPath = drm.detect_objects(str(binPath))

### Detection visualisation

In [ ]:
import trimesh
import numpy as np
import json

with open(detectionJsonPath) as f:
    data = json.load(f)

# Parameters
score_threshold = 0.8  # Only show boxes with score > threshold

# Colormap for labels
label_colors = [
    [1, 0, 0, 0.5],  # red, alpha 0.5
    [0, 1, 0, 0.5],  # green
    [0, 0, 1, 0.5],  # blue
    [1, 1, 0, 0.5],  # yellow
    [1, 0, 1, 0.5],  # magenta
    [0, 1, 1, 0.5],  # cyan
]

# Create list of meshes
meshes = []

for label, score, box in zip(data["labels_3d"], data["scores_3d"], data["bboxes_3d"]):
    if score < score_threshold:
        continue

    center = np.array(box[:3])
    size = np.array(box[3:6])
    rotation_z = box[6]
    color = label_colors[label % len(label_colors)]
    
    mesh = drm.create_trimesh_box(center, size, rotation_z, color)
    meshes.append(mesh)

pcd = drm.load_bin_pointcloud(str(binPath))
meshes.append(pcd)

# Combine meshes for visualization
scene = trimesh.Scene(meshes)

# Show interactive visualization
scene.show()

## Object cut out

In [ ]:
import numpy as np
from scipy.spatial.transform import Rotation as R
import os

def points_in_box(points, center, size, rotation_z):
    """
    Returns a boolean mask of points inside a 3D bounding box.
    Box is defined by center (x,y,z), size (l,w,h), and rotation around Z.
    """
    # Translate points to box frame
    points_local = points[:, :3] - center
    
    # Rotate points by -rotation_z
    rot = R.from_euler('z', -rotation_z).as_matrix()
    points_local = points_local @ rot.T
    
    # Check within box extents
    mask = (
        (points_local[:, 0] >= -size[0]/2) & (points_local[:, 0] <= size[0]/2) &
        (points_local[:, 1] >= -size[1]/2) & (points_local[:, 1] <= size[1]/2) &
        (points_local[:, 2] >= -size[2]/2) & (points_local[:, 2] <= size[2]/2)
    )
    return mask

def cut_pointcloud_by_boxes(points, labels, scores, boxes, score_threshold=0.5, out_dir="cut_points"):
    """
    Cut a point cloud into separate subsets per 3D bounding box.
    
    points: Nx6 array (x,y,z,r,g,b)
    labels: list of int labels
    scores: list of float scores
    boxes: Nx7 array (cx,cy,cz,l,w,h,rotation_z)
    score_threshold: ignore boxes below this score
    out_dir: folder to save each subset
    """
    os.makedirs(out_dir, exist_ok=True)
    points_remaining = points.copy()
    
    # Sort boxes by score descending
    sorted_idx = np.argsort(scores)[::-1]
    labels = [labels[i] for i in sorted_idx]
    scores = [scores[i] for i in sorted_idx]
    boxes = [boxes[i] for i in sorted_idx]
    
    saved_files = []
    
    for i, (label, score, box) in enumerate(zip(labels, scores, boxes)):
        if score < score_threshold:
            continue
        
        center = np.array(box[:3])
        size = np.array(box[3:6])
        rotation_z = box[6]
        
        mask = points_in_box(points_remaining, center, size, rotation_z)
        points_in = points_remaining[mask]
        
        if points_in.shape[0] == 0:
            continue
        
        # Save points inside this box
        filename = os.path.join(out_dir, f"box_{i}_label{label}.txt")
        np.savetxt(filename, points_in, fmt='%.6f')
        saved_files.append(filename)
        
        # Remove these points from remaining
        points_remaining = points_remaining[~mask]
    
    # Save remaining points as isolated
    if points_remaining.shape[0] > 0:
        isolated_file = os.path.join(out_dir, "isolated_points.txt")
        np.savetxt(isolated_file, points_remaining, fmt='%.6f')
        saved_files.append(isolated_file)
    
    print(f"Saved {len(saved_files)} pointclouds in {out_dir}")
    return saved_files

# Example usage:
points = np.loadtxt("pointcloud.txt")  # Nx6 xyzrgb
saved_files = cut_pointcloud_by_boxes(points, data["labels_3d"], data["scores_3d"], data["bboxes_3d"])